# 翻译 (PyTorch)

Install the Transformers, Datasets, and Evaluate libraries to run this notebook.

In [ ]:
!pip install datasets evaluate transformers[sentencepiece]
!pip install accelerate
# To run the training on TPU, you will need to uncomment the following line:
# !pip install cloud-tpu-client==0.10 torch==1.9.0 https://storage.googleapis.com/tpu-pytorch/wheels/torch_xla-1.9-cp37-cp37m-linux_x86_64.whl
!apt install git-lfs

You will need to setup git, adapt your email and name in the following cell.

In [ ]:
!git config --global user.email "you@example.com"
!git config --global user.name "Your Name"

You will also need to be logged in to the Hugging Face Hub. Execute the following and enter your credentials.

In [8]:
from huggingface_hub import notebook_login

notebook_login()

Task was destroyed but it is pending!
task: <Task pending name='Task-126' coro=<_async_in_context.<locals>.run_in_context() running at /home/work/miniforge3/envs/llm-hf/lib/python3.11/site-packages/ipykernel/utils.py:57> cb=[ZMQStream._run_callback.<locals>._log_error() at /home/work/miniforge3/envs/llm-hf/lib/python3.11/site-packages/zmq/eventloop/zmqstream.py:563]>
/home/work/miniforge3/envs/llm-hf/lib/python3.11/asyncio/base_events.py:679: RuntimeWarning: coroutine '_async_in_context.<locals>.run_in_context' was never awaited
  self._ready.clear()


In [15]:
from datasets import load_dataset
import evaluate

raw_datasets = load_dataset("ArmelRandy/kde4")

In [44]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['en', 'fr'],
        num_rows: 20058
    })
})

In [17]:
split_datasets = raw_datasets["train"].train_test_split(train_size=0.9, seed=20)
split_datasets

DatasetDict({
    train: Dataset({
        features: ['en', 'fr'],
        num_rows: 18052
    })
    test: Dataset({
        features: ['en', 'fr'],
        num_rows: 2006
    })
})

In [18]:
split_datasets["validation"] = split_datasets.pop("test")

In [19]:
split_datasets["train"][1]

{'en': 'The PERMUT() function returns the number of permutations. The first parameter is the number of elements, and the second parameter is the number of elements used in the permutation.',
 'fr': "La fonction PERMUT() renvoie le nombre de permutations. Le premier paramètre est le nombre d'éléments et le second est le nombre d'éléments à permuter."}

In [22]:
import transformers; print(transformers.__version__)

5.3.0


In [23]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_checkpoint = "Helsinki-NLP/opus-mt-en-fr"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

def translate(texts):
    inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        outputs = model.generate(**inputs)
    return [{"translation_text": t} for t in tokenizer.batch_decode(outputs, skip_special_tokens=True)]

translate("Default to expanded threads")

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/778k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

/home/work/miniforge3/envs/llm-hf/lib/python3.11/site-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


...

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/301M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

[{'translation_text': 'Par défaut pour les threads élargis'}]

In [24]:
split_datasets["train"][172]

{'en': 'For the SMB protocol to work, it is required to have Samba correctly installed. If you have an NT domain controller, you will need at least Samba version 2.0 or higher. If you want to access & Windows; 2000 shares, you will need Samba version 2.0.7 or higher. Older versions may work too, but have not been tested.',
 'fr': "Pour que le protocole SMB fonctionne, il est nécessaire d'avoir Samba correctement installé. Si vous avec un contrôleur de domaine NT, vous aurez besoin d'au moins Samba version 2.0 ou plus. Si vous voulez accéder aux partages & Windows; 2000, vous aurez besoin de Samba version 2.0.7 ou plus. Les versions plus anciennes peuvent fonctionner mais n'ont pas été testées."}

In [27]:
translate(
    split_datasets["train"][172]['en']
)

[{'translation_text': "Pour que le protocole SMB fonctionne, il faut que Samba soit correctement installé. Si vous avez un contrôleur de domaine NT, vous aurez besoin d'au moins la version 2.0 de Samba ou plus. Si vous souhaitez accéder aux actions de & Windows; 2000 vous aurez besoin de la version 2.0.7 de Samba ou plus. Les versions plus anciennes peuvent aussi fonctionner, mais n'ont pas été testées."}]

In [28]:
from transformers import AutoTokenizer

model_checkpoint = "Helsinki-NLP/opus-mt-en-fr"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, return_tensors="pt")

In [32]:
en_sentence = split_datasets["train"][1]["en"]
fr_sentence = split_datasets["train"][1]["fr"]

model_inputs = tokenizer(en_sentence, text_target=fr_sentence, truncation=True)
model_inputs

{'input_ids': [35, 10460, 559, 7075, 401, 28, 3318, 11097, 4, 365, 7, 329, 18888, 2252, 3, 35, 293, 25352, 32, 4, 365, 7, 2720, 2, 10, 4, 787, 25352, 32, 4, 365, 7, 2720, 403, 18, 4, 329, 18888, 446, 3, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [80, 952, 10460, 559, 7075, 401, 28, 19730, 19, 384, 5, 329, 18888, 2252, 3, 60, 698, 23408, 43, 19, 384, 20, 6, 13635, 11, 19, 787, 43, 19, 384, 20, 6, 13635, 17, 329, 18888, 108, 3, 0]}

In [36]:
wrong_targets = tokenizer(fr_sentence)
print(f'en_sentence: {en_sentence}')
print(f'fr_sentence: {fr_sentence}')
print(tokenizer.convert_ids_to_tokens(wrong_targets["input_ids"]))
print(tokenizer.convert_ids_to_tokens(model_inputs["labels"]))

en_sentence: The PERMUT() function returns the number of permutations. The first parameter is the number of elements, and the second parameter is the number of elements used in the permutation.
fr_sentence: La fonction PERMUT() renvoie le nombre de permutations. Le premier paramètre est le nombre d'éléments et le second est le nombre d'éléments à permuter.
['▁La', '▁f', 'on', 'ction', '▁PER', 'M', 'UT', '(', ')', '▁re', 'n', 'vo', 'ie', '▁le', '▁nombre', '▁de', '▁per', 'mut', 'ations', '.', '▁Le', '▁premier', '▁para', 'm', 'è', 't', 're', '▁est', '▁le', '▁nombre', '▁d', "'", 'élé', 'ments', '▁et', '▁le', '▁second', '▁est', '▁le', '▁nombre', '▁d', "'", 'élé', 'ments', '▁à', '▁per', 'm', 'uter', '.', '</s>']
['▁La', '▁fonction', '▁PER', 'M', 'UT', '(', ')', '▁renvoie', '▁le', '▁nombre', '▁de', '▁per', 'mut', 'ations', '.', '▁Le', '▁premier', '▁paramètre', '▁est', '▁le', '▁nombre', '▁d', "'", 'éléments', '▁et', '▁le', '▁second', '▁est', '▁le', '▁nombre', '▁d', "'", 'éléments', '▁à', '▁per

In [53]:
max_input_length = 128
max_target_length = 128


def preprocess_function(examples):
    model_inputs = tokenizer(examples['en'], text_target=examples['fr'], max_length=max_input_length, truncation=True)

    return model_inputs

In [54]:
preprocess_function(split_datasets['train'][:2])

{'input_ids': [[1111, 1097, 4, 25129, 27346, 26, 1829, 4272, 2, 32102, 202, 45, 10887, 14096, 111, 46, 23522, 30391, 4, 22635, 791, 5772, 4, 402, 89, 75, 13259, 50, 9319, 3, 84, 8117, 746, 2805, 32, 121, 494, 46, 349, 21, 23298, 101, 30, 4, 1480, 2, 85, 3707, 32102, 12, 45, 18919, 57, 7373, 3, 0], [35, 10460, 559, 7075, 401, 28, 3318, 11097, 4, 365, 7, 329, 18888, 2252, 3, 35, 293, 25352, 32, 4, 365, 7, 2720, 2, 10, 4, 787, 25352, 32, 4, 365, 7, 2720, 403, 18, 4, 329, 18888, 446, 3, 0]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], 'labels': [[2893, 88, 12500, 87, 6, 21975, 20, 6, 2133, 27, 16, 8333, 4576, 2, 16, 2728, 388, 100, 906, 1088, 14515, 1838, 23, 46558, 8, 15527, 3724, 29, 88, 32640, 36, 8, 6584, 402, 89, 7101,

In [55]:
tokenized_datasets = split_datasets.map(
    preprocess_function,
    batched=True,
    remove_columns=split_datasets["train"].column_names,
)

Map:   0%|          | 0/18052 [00:00<?, ? examples/s]

Map:   0%|          | 0/2006 [00:00<?, ? examples/s]

In [56]:
from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [57]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

In [58]:
batch = data_collator([tokenized_datasets["train"][i] for i in range(1, 3)])
batch.keys()

KeysView({'input_ids': tensor([[   35, 10460,   559,  7075,   401,    28,  3318, 11097,     4,   365,
             7,   329, 18888,  2252,     3,    35,   293, 25352,    32,     4,
           365,     7,  2720,     2,    10,     4,   787, 25352,    32,     4,
           365,     7,  2720,   403,    18,     4,   329, 18888,   446,     3,
             0],
        [  560,    69,    77,  6443, 34055,     4,   359,  6199, 13415,  6442,
          2128,    21, 26415,    26,   301,   331,    37,   301,   548,   213,
          1298,   639,    12, 26518,    58,   309,   529,   309, 17770,   301,
           602,   149,    30,     4,  6467,  1410,     3,     0, 59513, 59513,
         59513]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0]]), 'labe

In [59]:
batch["labels"]

tensor([[   80,   952, 10460,   559,  7075,   401,    28, 19730,    19,   384,
             5,   329, 18888,  2252,     3,    60,   698, 23408,    43,    19,
           384,    20,     6, 13635,    11,    19,   787,    43,    19,   384,
            20,     6, 13635,    17,   329, 18888,   108,     3,     0,  -100,
          -100,  -100,  -100,  -100,  -100,  -100,  -100],
        [  104,   167,    15,   871,    38,  6502,   614,    14,     6,  2569,
            22,   767,  8810,     5,     8,  6442,   359,  6199, 13415,    27,
           301,   331,   402, 38492,   301,   548,   344, 12754, 11979,   153,
           402, 29033,   954,   529,   309, 17770,   301,   602,   402, 29033,
           416,    23,   863,     5,  3745,     3,     0]])

In [60]:
batch["decoder_input_ids"]

tensor([[59513,    80,   952, 10460,   559,  7075,   401,    28, 19730,    19,
           384,     5,   329, 18888,  2252,     3,    60,   698, 23408,    43,
            19,   384,    20,     6, 13635,    11,    19,   787,    43,    19,
           384,    20,     6, 13635,    17,   329, 18888,   108,     3,     0,
         59513, 59513, 59513, 59513, 59513, 59513, 59513],
        [59513,   104,   167,    15,   871,    38,  6502,   614,    14,     6,
          2569,    22,   767,  8810,     5,     8,  6442,   359,  6199, 13415,
            27,   301,   331,   402, 38492,   301,   548,   344, 12754, 11979,
           153,   402, 29033,   954,   529,   309, 17770,   301,   602,   402,
         29033,   416,    23,   863,     5,  3745,     3]])

In [61]:
for i in range(1, 3):
    print(tokenized_datasets["train"][i]["labels"])

[80, 952, 10460, 559, 7075, 401, 28, 19730, 19, 384, 5, 329, 18888, 2252, 3, 60, 698, 23408, 43, 19, 384, 20, 6, 13635, 11, 19, 787, 43, 19, 384, 20, 6, 13635, 17, 329, 18888, 108, 3, 0]
[104, 167, 15, 871, 38, 6502, 614, 14, 6, 2569, 22, 767, 8810, 5, 8, 6442, 359, 6199, 13415, 27, 301, 331, 402, 38492, 301, 548, 344, 12754, 11979, 153, 402, 29033, 954, 529, 309, 17770, 301, 602, 402, 29033, 416, 23, 863, 5, 3745, 3, 0]


In [ ]:
!pip install sacrebleu

In [62]:
import evaluate

metric = evaluate.load("sacrebleu")

In [63]:
predictions = [
    "This plugin lets you translate web pages between several languages automatically."
]
references = [
    [
        "This plugin allows you to automatically translate web pages between several languages."
    ]
]
metric.compute(predictions=predictions, references=references)

{'score': 46.750469682990165,
 'counts': [11, 6, 4, 3],
 'totals': [12, 11, 10, 9],
 'precisions': [91.66666666666667,
  54.54545454545455,
  40.0,
  33.333333333333336],
 'bp': 0.9200444146293233,
 'sys_len': 12,
 'ref_len': 13}

In [64]:
predictions = ["This This This This"]
references = [
    [
        "This plugin allows you to automatically translate web pages between several languages."
    ]
]
metric.compute(predictions=predictions, references=references)

{'score': 1.683602693167689,
 'counts': [1, 0, 0, 0],
 'totals': [4, 3, 2, 1],
 'precisions': [25.0, 16.666666666666668, 12.5, 12.5],
 'bp': 0.10539922456186433,
 'sys_len': 4,
 'ref_len': 13}

In [65]:
predictions = ["This plugin"]
references = [
    [
        "This plugin allows you to automatically translate web pages between several languages."
    ]
]
metric.compute(predictions=predictions, references=references)

{'score': 0.0,
 'counts': [2, 1, 0, 0],
 'totals': [2, 1, 0, 0],
 'precisions': [100.0, 100.0, 0.0, 0.0],
 'bp': 0.004086771438464067,
 'sys_len': 2,
 'ref_len': 13}

In [66]:
import numpy as np


def compute_metrics(eval_preds):
    preds, labels = eval_preds
    # In case the model returns more than the prediction logits
    if isinstance(preds, tuple):
        preds = preds[0]

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    # Replace -100s in the labels as we can't decode them
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Some simple post-processing
    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [[label.strip()] for label in decoded_labels]

    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    return {"bleu": result["score"]}

In [67]:
from huggingface_hub import notebook_login

notebook_login()

In [68]:
from transformers import Seq2SeqTrainingArguments

args = Seq2SeqTrainingArguments(
    f"marian-finetuned-kde4-en-to-fr",
    eval_strategy="no",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=True,
    push_to_hub=True,
)

In [69]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model,
    args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

In [72]:
from transformers.utils.notebook import NotebookProgressCallback

In [73]:
trainer.remove_callback(NotebookProgressCallback)
trainer.evaluate(max_length=max_target_length)

{'eval_loss': 1.1564863920211792,
 'eval_model_preparation_time': 0.0021,
 'eval_bleu': 41.7653462500669,
 'eval_runtime': 50.7139,
 'eval_samples_per_second': 39.555,
 'eval_steps_per_second': 0.631,
 'epoch': 0}

In [74]:
trainer.add_callback(NotebookProgressCallback)
trainer.train()
trainer.remove_callback(NotebookProgressCallback)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Step,Training Loss
500,1.010697
1000,0.893828
1500,0.851380


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [75]:
trainer.evaluate(max_length=max_target_length)

{'eval_loss': 0.8744521141052246,
 'eval_model_preparation_time': 0.0021,
 'eval_bleu': 49.69496061477768,
 'eval_runtime': 49.2924,
 'eval_samples_per_second': 40.696,
 'eval_steps_per_second': 0.649,
 'epoch': 3.0}

In [ ]:
trainer.push_to_hub(tags="translation", commit_message="Training complete")

'https://huggingface.co/sgugger/marian-finetuned-kde4-en-to-fr/commit/3601d621e3baae2bc63d3311452535f8f58f6ef3'

In [ ]:
from torch.utils.data import DataLoader

tokenized_datasets.set_format("torch")
train_dataloader = DataLoader(
    tokenized_datasets["train"],
    shuffle=True,
    collate_fn=data_collator,
    batch_size=8,
)
eval_dataloader = DataLoader(
    tokenized_datasets["validation"], collate_fn=data_collator, batch_size=8
)

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

In [ ]:
from transformers import AdamW

optimizer = AdamW(model.parameters(), lr=2e-5)

In [ ]:
from accelerate import Accelerator

accelerator = Accelerator()
model, optimizer, train_dataloader, eval_dataloader = accelerator.prepare(
    model, optimizer, train_dataloader, eval_dataloader
)

In [ ]:
from transformers import get_scheduler

num_train_epochs = 3
num_update_steps_per_epoch = len(train_dataloader)
num_training_steps = num_train_epochs * num_update_steps_per_epoch

lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)

In [ ]:
from huggingface_hub import HfApi, get_full_repo_name

model_name = "marian-finetuned-kde4-en-to-fr-accelerate"
repo_name = get_full_repo_name(model_name)
repo_name

'sgugger/marian-finetuned-kde4-en-to-fr-accelerate'

In [ ]:
output_dir = "marian-finetuned-kde4-en-to-fr-accelerate"
api = HfApi()
api.create_repo(repo_name, exist_ok=True)

In [ ]:
def postprocess(predictions, labels):
    predictions = predictions.cpu().numpy()
    labels = labels.cpu().numpy()

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)

    # Replace -100 in the labels as we can't decode them.
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Some simple post-processing
    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [[label.strip()] for label in decoded_labels]
    return decoded_preds, decoded_labels

In [ ]:
from tqdm.auto import tqdm
import torch

progress_bar = tqdm(range(num_training_steps))

for epoch in range(num_train_epochs):
    # Training
    model.train()
    for batch in train_dataloader:
        outputs = model(**batch)
        loss = outputs.loss
        accelerator.backward(loss)

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)

    # Evaluation
    model.eval()
    for batch in tqdm(eval_dataloader):
        with torch.no_grad():
            generated_tokens = accelerator.unwrap_model(model).generate(
                batch["input_ids"],
                attention_mask=batch["attention_mask"],
                max_length=128,
            )
        labels = batch["labels"]

        # Necessary to pad predictions and labels for being gathered
        generated_tokens = accelerator.pad_across_processes(
            generated_tokens, dim=1, pad_index=tokenizer.pad_token_id
        )
        labels = accelerator.pad_across_processes(labels, dim=1, pad_index=-100)

        predictions_gathered = accelerator.gather(generated_tokens)
        labels_gathered = accelerator.gather(labels)

        decoded_preds, decoded_labels = postprocess(predictions_gathered, labels_gathered)
        metric.add_batch(predictions=decoded_preds, references=decoded_labels)

    results = metric.compute()
    print(f"epoch {epoch}, BLEU score: {results['score']:.2f}")

    # Save and upload
    accelerator.wait_for_everyone()
    unwrapped_model = accelerator.unwrap_model(model)
    unwrapped_model.save_pretrained(output_dir, save_function=accelerator.save)
    if accelerator.is_main_process:
        tokenizer.save_pretrained(output_dir)
        api.upload_folder(
            repo_id=repo_name,
            folder_path=output_dir,
            commit_message=f"Training in progress epoch {epoch}",
        )

epoch 0, BLEU score: 53.47
epoch 1, BLEU score: 54.24
epoch 2, BLEU score: 54.44

In [ ]:
from transformers import pipeline

# Replace this with your own checkpoint
model_checkpoint = "huggingface-course/marian-finetuned-kde4-en-to-fr"
translator = pipeline("translation", model=model_checkpoint)
translator("Default to expanded threads")

[{'translation_text': 'Par défaut, développer les fils de discussion'}]

In [ ]:
translator(
    "Unable to import %1 using the OFX importer plugin. This file is not the correct format."
)

[{'translation_text': "Impossible d'importer %1 en utilisant le module externe d'importation OFX. Ce fichier n'est pas le bon format."}]